# RSNA Knee — submission (inference only, Internet OFF)
学習は `02_cnn_embed` で行い、保存した重みとモデルをKaggle Datasetから読む。`src` も Dataset から。

In [ ]:
import os, sys, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch, joblib
from pathlib import Path

def find(name):
    # Kaggleのマウント位置(/kaggle/input/<slug> または .../datasets/<user>/<slug>)の違いを吸収
    for base in [Path('/kaggle/input'), Path('/kaggle/input/datasets')]:
        if not base.exists(): continue
        for d in base.iterdir():
            if d.name == name: return d
            if d.is_dir() and d.name not in ('competitions', 'datasets'):
                for s in d.iterdir():
                    if s.name == name: return s
    raise FileNotFoundError(f'{name} not found. /kaggle/input: {[p.name for p in Path("/kaggle/input").iterdir()]}')

SRC, MODEL = find('rsna-knee-src'), find('rsna-knee-model')
sys.path.insert(0, str(SRC))
import cnn_features as cf
from pseudo_labels import LABELS
_name = 'rsna-knee-abnormality-detection'
COMP = next((d for d in [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name] if (d/'test.csv').exists()), None)
assert COMP, 'test.csv not found'
print('COMP', COMP, '| MODEL', MODEL)

In [ ]:
test = pd.read_csv(COMP/'test.csv'); teseries = pd.read_csv(COMP/'test_series.csv')
print(test.shape, teseries.shape)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
import tempfile
CACHE = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path(tempfile.gettempdir())
FTS = sorted(MODEL.glob('ft_model*.pt'))
if FTS:                                      # ファインチューニング版(全モデルの予測を平均。設定(size/k/center)が違うモデルはキャッシュを別に作る)
    import finetune as ft
    groups = {}
    for f in FTS:
        ck = torch.load(f, map_location='cpu')
        groups.setdefault((ck['size'], ck['k'], ck.get('center', 1.0), ck.get('crop', 1.0)), []).append((f, ck))
    preds, names = [], []
    for gi, ((size, k, center, crop), items) in enumerate(groups.items()):
        X, M, META = ft.cache_volumes(test, teseries, COMP/'test_series', size, k, CACHE, f'test{gi}', center=center, crop=crop)
        for f, ck in items:
            net = ft.KneeNet(len(LABELS), pretrained=False, meta_dim=ft.META_DIM if ck.get('meta') else 0,
                              adj=ck.get('adj', False), arch=ck.get('arch', 'resnet18'))
            net.load_state_dict(ck['state']); net.to(dev)
            preds.append(ft.predict(net, X, M, np.arange(len(test)), dev, META=META)); names.append(f.name); del net
        del X
    print('models averaged (%d groups):' % len(groups), names)
    P = np.mean(preds, axis=0)
    sub = pd.DataFrame(P, columns=LABELS); sub.insert(0, 'StudyInstanceUID', test['StudyInstanceUID'].values)
else:                                        # 埋め込み+LightGBM版
    model = cf.load_model(dev, weights_path=MODEL/'resnet18_imagenet.pth')
    models = joblib.load(MODEL/'models.joblib')
    Xte = cf.extract(test, teseries, COMP/'test_series', model, dev)
    import cv_eval
    sub = cv_eval.predict_models(models, Xte, LABELS, test['StudyInstanceUID'])

In [ ]:
assert sub.shape[0] == len(test) and sub[LABELS].notna().all().all()
sub.to_csv('submission.csv', index=False); print(sub.shape); sub.head()